# Experiment 0 — Dataset geometry (model-free)

**Question.** Before we read anything out of a model: what structure does the dataset *itself*
contain, and how much of it is trivially recoverable from the surface text? Every claim in
experiment 3 ("this tag shows up as geometry in activation space") is only interesting relative
to two references — the **surface floor** (what a bag of words already knows) and the **target
geometry** (what structure the ground-truth labels actually have). This notebook measures both.

**Why it comes first.** Shivam's caveat on the PCA experiment was that the input data must
contain genuine order, because a poorly curated dataset shows nothing regardless of what the
model represents. The converse failure is just as bad: a dataset where the tag is a deterministic
function of the surface text will show beautiful structure that means nothing. This notebook
catches both, costs no GPU, and runs in seconds.

**Design.** Exactly the dataset of experiment 3 — the same seed, the same pairs, the same six
deterministic surface forms per problem — represented four model-free ways:

| representation | what it captures | role |
| --- | --- | --- |
| word TF-IDF | content-word overlap | surface floor |
| char n-gram TF-IDF | variable names, operator glyphs, morphology | surface floor (catches what word tokens miss, esp. RG) |
| word count (1-D) | length only | the confound most likely to masquerade as complexity |
| symbolic subtree bag | the law's own term structure | target geometry |

**Sections.** **A** register/form, **B** complexity, **C** law identity, **D** the target
geometry itself, **E** exports for experiment 3. Each section ends with the number that
experiment 3 has to beat.

**Runtime.** CPU, ~1 minute, no model download. (Section C has an optional stock-sentence-encoder
rung, off by default.)

In [ ]:
# ------------------------------ Configuration ------------------------------
# The dataset block must stay identical to experiment 03, or the floors
# computed here do not apply to the activations measured there.

REPO_URL = "https://github.com/shivam-raval96/semantic-drift-autoformalization.git"
REPO_COMMIT = "c9e128f247b6daff5b47fd9711ded5e02d1095e9"  # same pin as experiments 01 and 03

SEED = 0
PER_BIN = 10
STORY_THEMES = ["graft", "paint", "signal", "tea"]

# Character n-grams: wide enough to span "v a b" style RG variable runs.
CHAR_NGRAM_RANGE = (3, 5)

# Optional fifth rung: a stock sentence encoder, as an "is this specific to the
# reasoning model?" control. Left off by default so the notebook stays model-free;
# set to e.g. "sentence-transformers/all-MiniLM-L6-v2" to include it.
SENTENCE_ENCODER = None

N_SHOW = 8       # laws highlighted in the scatters
QUICK_TEST = False
if QUICK_TEST:
    PER_BIN = 2

In [ ]:
# ------------------------- Environment and outputs -------------------------

# No -U: Colab already ships all three, and force-upgrading scipy/numpy under a
# live runtime is the usual cause of "numpy.dtype size changed" on first run.
%pip install -q scikit-learn matplotlib scipy

import hashlib
import json
import subprocess
import sys
from pathlib import Path

if not Path("semantic-drift-autoformalization").exists():
    subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
subprocess.run(
    ["git", "-C", "semantic-drift-autoformalization", "checkout", "-q", REPO_COMMIT],
    check=True,
)
sys.path.insert(0, str(Path("semantic-drift-autoformalization/informalizing-etp").resolve()))

from benchmark import drop_vacuous, load_equations, sample_pairs_stratified, synthesize_response
from literalform import render_description
from storyform import Var, parse_equation, render_story

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer

OUT_DIR = Path("exp00-outputs")
try:
    from google.colab import drive

    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/mech-interp-experiments/exp00-dataset-geometry")
except Exception as error:
    print(f"Google Drive not available ({error}); using local {OUT_DIR}/")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("outputs ->", OUT_DIR.resolve())

## The dataset

Identical construction to experiment 3: complexity-stratified sample of implication pairs
(vacuous laws dropped), each rendered six ways by deterministic code — four themed stories with
near-disjoint vocabularies, one literal-NL description, one Rigid-Grammar rendering. Tags
(`pair_id`, form, theme, `ops_total`, `depth`) are ground truth by construction.

The printed `texts SHA` is the handle that ties the two notebooks together: if it differs from
the one experiment 3 prints, the two are not looking at the same dataset and the floors below do
not transfer.

In [ ]:
# ------------------------------ Build the dataset ------------------------------

equations, equations_sha = load_equations()
pairs = drop_vacuous(sample_pairs_stratified(equations, PER_BIN, SEED, form="story"))
print(f"ETP equation list: {len(equations)} equations (sha256 {equations_sha[:12]})")
print(f"{len(pairs)} pairs after dropping vacuous laws")

records, kept_pairs, dropped = [], [], []
for sample in pairs:
    meta = sample["metadata"]
    tags = {"pair_id": sample["pair_id"], "ops_total": sample["ops_total"],
            "depth": sample["depth"]}
    try:
        rendered = [
            {**tags, "form": "story", "theme": theme,
             "text": render_story(meta["equation_e"], meta["equation_f"], theme_key=theme)[0]}
            for theme in STORY_THEMES
        ]
        rendered.append({**tags, "form": "literal", "theme": "literal",
                         "text": render_description(meta["equation_e"], meta["equation_f"])[0]})
        rendered.append({**tags, "form": "rg", "theme": "rg",
                         "text": synthesize_response(meta)})
    except (ValueError, KeyError) as error:
        dropped.append((sample["pair_id"], str(error)))  # keep the 6-form grid complete
        continue
    records.extend(rendered)
    kept_pairs.append(sample)

texts = [r["text"] for r in records]
forms = np.array([r["form"] for r in records])
themes = np.array([r["theme"] for r in records])
surfaces = themes                                     # the six surface groups
pair_ids = np.array([r["pair_id"] for r in records])
ops = np.array([r["ops_total"] for r in records])
depth = np.array([r["depth"] for r in records])
lengths = np.array([len(t.split()) for t in texts], dtype=float)

law_order = [s["pair_id"] for s in kept_pairs]
n_pairs = len(law_order)
CHANCE = 1 / n_pairs

print(f"{n_pairs} pairs x 6 forms = {len(records)} texts"
      + (f"  ({len(dropped)} pairs dropped: {dropped})" if dropped else ""))
print(f"texts SHA (must match experiment 03): "
      f"{hashlib.sha256(json.dumps(texts).encode()).hexdigest()[:12]}")
print(f"ops_total: {sorted(np.unique(ops).tolist())}   depth: {sorted(np.unique(depth).tolist())}")

## Model-free representations

Four featurizations of the same 6-form dataset. The first three are surface floors computed from
the raw strings; the fourth is computed from the *terms* and is the target geometry, lifted from
laws to texts by giving every rendering of a law that law's symbolic vector.

The symbolic representation is a **bag of alpha-renamed subtrees**: every subtree of E's and F's
two sides, with variables renamed in first-occurrence order, so `(a∘(b∘a))` and `(y∘(z∘y))` share
a key while `(a∘(b∘a))` and `(a∘(b∘c))` do not. Subtrees are prefixed by which equation they came
from (`E:` = ASSUME, `F:` = ASK). Bare-variable leaves are dropped and features are IDF-weighted:
without that, every law shares the leaf features and all pairwise cosines collapse to ~0.9,
which hides the real structure.

In [ ]:
# ---------------------- Representations ----------------------

def alpha_shape(term):
    """Canonical string for a term with variables renamed by first occurrence,
    so structurally identical subtrees share a key whatever letters they use."""
    names = {}

    def go(t):
        if isinstance(t, Var):
            return names.setdefault(t.name, f"x{len(names)}")
        return f"({go(t.left)}.{go(t.right)})"

    return go(term)


def subtrees(term):
    yield term
    if not isinstance(term, Var):
        yield from subtrees(term.left)
        yield from subtrees(term.right)


def subtree_bag(meta, drop_leaves=True):
    bag = {}
    for side, key in (("E", "equation_e"), ("F", "equation_f")):
        for root in parse_equation(meta[key]):
            for sub in subtrees(root):
                if drop_leaves and isinstance(sub, Var):
                    continue
                k = f"{side}:{alpha_shape(sub)}"
                bag[k] = bag.get(k, 0) + 1
    return bag


def l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)


bags = [subtree_bag(s["metadata"]) for s in kept_pairs]
subtree_keys = sorted({k for b in bags for k in b})
LAW_SYMBOLIC = np.array([[b.get(k, 0) for k in subtree_keys] for b in bags], dtype=float)
LAW_SYMBOLIC *= np.log((1 + n_pairs) / (1 + (LAW_SYMBOLIC > 0).sum(axis=0)))  # IDF
LAW_SYMBOLIC = l2(LAW_SYMBOLIC)

law_row = {p: i for i, p in enumerate(law_order)}
law_index = np.array([law_row[p] for p in pair_ids])

REPS = {
    "word TF-IDF": l2(np.asarray(TfidfVectorizer().fit_transform(texts).todense())),
    "char TF-IDF": l2(np.asarray(TfidfVectorizer(
        analyzer="char_wb", ngram_range=CHAR_NGRAM_RANGE).fit_transform(texts).todense())),
    "length only": lengths.reshape(-1, 1).copy(),
    "symbolic": LAW_SYMBOLIC[law_index],   # target geometry, lifted to texts
}

if SENTENCE_ENCODER:
    from sentence_transformers import SentenceTransformer

    REPS["sentence encoder"] = l2(
        SentenceTransformer(SENTENCE_ENCODER).encode(texts, convert_to_numpy=True))

TEXT_REPS = [n for n in REPS if n not in ("symbolic", "length only")]

print(f"{len(subtree_keys)} distinct subtree features over {n_pairs} laws")
for name, X in REPS.items():
    print(f"  {name:18s} {X.shape}")

In [ ]:
# ---------------------- Shared metrics (same definitions as experiment 03) ----------------------

def cosine_sims(X):
    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
    return Xn @ Xn.T


def similarity(X):
    """Cosine for multi-dimensional reps; negative distance for the 1-D length
    rep, where every vector is collinear and cosine is uninformative."""
    if X.shape[1] == 1:
        v = X[:, 0]
        return -np.abs(v[:, None] - v[None, :])
    return cosine_sims(X)


def law_retrieval_all(X):
    """1-NN law retrieval: the nearest neighbour among texts of a *different*
    surface (another theme, or another form) must encode the same law."""
    sims = similarity(X).copy()
    sims[surfaces[:, None] == surfaces[None, :]] = -np.inf   # also removes self
    return float((pair_ids[sims.argmax(axis=1)] == pair_ids).mean())


def law_retrieval_story(X):
    """Story-only variant: candidates restricted to stories of other themes,
    whose vocabularies are near-disjoint by construction."""
    sims = similarity(X).copy()
    story = forms == "story"
    allowed = story[None, :] & (themes[:, None] != themes[None, :])
    sims[~allowed] = -np.inf
    return float((pair_ids[sims.argmax(axis=1)] == pair_ids)[story].mean())


def pca2(X):
    p = PCA(n_components=2).fit(X)
    return p.transform(X), float(p.explained_variance_ratio_.sum())


def residualize(X, covariate):
    """Least-squares removal of a scalar covariate (used for length) from every
    feature dimension. Experiment 03 can apply this to activations verbatim."""
    A = np.stack([np.ones_like(covariate), covariate], axis=1)
    beta, *_ = np.linalg.lstsq(A, X, rcond=None)
    return X - A @ beta

## A — Register / form: separation is free, ordering is not

Two questions that the activation version of this plot cannot tell apart.

1. *Do the forms separate?* If a bag of words already splits story / literal / RG, then finding
   the same split in activations says nothing about the model.
2. *Do they separate in ladder order* — does literal sit **between** story and RG? That is
   experiment 7's H1 in its weakest form, and it is not implied by (1): three groups can be
   mutually distinct while forming a triangle rather than a path.

The number below is the ladder statistic: project the literal centroid onto the story→RG axis
(t = 0 at story, t = 1 at RG) and report the off-axis residual as a fraction of the axis length.
Betweenness means t ≈ 0.5 *and* a small residual. A large residual is the null that activations
would have to beat.

In [ ]:
# ---------------------- A: form separation and ladder ordering ----------------------

SURFACE_STYLE = {
    "graft": ("#8c564b", "o"), "paint": ("#e377c2", "o"),
    "signal": ("#1f77b4", "o"), "tea": ("#ff7f0e", "o"),
    "literal": ("#2ca02c", "^"), "rg": ("#000000", "s"),
}
PLOT_REPS = [n for n in REPS if n != "length only"]

fig, axes = plt.subplots(1, len(PLOT_REPS), figsize=(4.2 * len(PLOT_REPS), 4.2), squeeze=False)
fig.suptitle("Data-space PCA by surface form (no model involved)")
for ax, name in zip(axes[0], PLOT_REPS):
    X2, evr = pca2(REPS[name])
    for key, (color, marker) in SURFACE_STYLE.items():
        m = surfaces == key
        ax.scatter(X2[m, 0], X2[m, 1], s=12, c=color, marker=marker, alpha=0.7, label=key)
    centroids = [X2[forms == f].mean(axis=0) for f in ("story", "literal", "rg")]
    ax.plot(*zip(*centroids), "k--", lw=1, zorder=3)
    for (x, y), label in zip(centroids, ("S", "L", "RG")):
        ax.annotate(label, (x, y), fontsize=11, fontweight="bold", zorder=4)
    ax.set_title(f"{name}  ({evr:.0%} var in 2D)", fontsize=9)
    ax.set_xticks([])
    ax.set_yticks([])
axes[0][0].legend(fontsize=7, loc="best")
plt.tight_layout()
plt.show()

def ladder_statistic(X):
    cS, cL, cR = (X[forms == f].mean(axis=0) for f in ("story", "literal", "rg"))
    axis = cR - cS
    t = float(axis @ (cL - cS) / (axis @ axis))
    return {"t": t,
            "off_axis_residual": float(np.linalg.norm((cL - cS) - t * axis)
                                       / np.linalg.norm(axis))}


print("ladder statistic: literal centroid on the story->RG axis")
ladder = {name: ladder_statistic(REPS[name]) for name in TEXT_REPS}
for name, s in ladder.items():
    verdict = "ordered path" if 0.2 < s["t"] < 0.8 and s["off_axis_residual"] < 0.35 else "not a path"
    print(f"  {name:18s} t = {s['t']:+.2f}   "
          f"off-axis residual = {s['off_axis_residual']:.2f} x |axis|   -> {verdict}")

# The statistic is only defined for representations that can distinguish the forms.
# It is undefined for "length only" (all three centroids are collinear in 1-D, so the
# residual is 0 by arithmetic) and degenerate for "symbolic" (a law's literal and RG
# renderings carry the *same* symbolic vector, so t = 1 and residual = 0 exactly) --
# printed here only as a check that the symbolic rep is form-invariant as intended.
sanity = ladder_statistic(REPS["symbolic"])
print(f"  {'symbolic (check)':18s} t = {sanity['t']:+.2f}   "
      f"residual = {sanity['off_axis_residual']:.2f}   "
      f"<- must be exactly +1.00 / 0.00: symbolic rep is form-invariant")

## B — Complexity: the confound check that decides whether the analysis is worth running

`ops_total` is the number of operations in the law, and the renderers are templates — one clause
per operation. So the first thing to check is not whether complexity appears in activation space
but whether it is simply **text length**. If it is, a "complexity gradient" at any layer is a
length gradient, and the honest fix is upstream (length-matched instances) rather than in the
plotting.

Below: the correlation within each form, then the same PCA colored by `ops_total`, then the same
plot after regressing word count out of the representation. Whatever survives residualization is
the part of the complexity signal that is not length.

In [ ]:
# ---------------------- B: is complexity just length? ----------------------

print("corr(word count, ops_total), computed within each form so form is not the driver:")
length_confound = {}
for f in ("story", "literal", "rg"):
    m = forms == f
    r_ops = float(np.corrcoef(lengths[m], ops[m])[0, 1])
    r_depth = float(np.corrcoef(lengths[m], depth[m])[0, 1])
    length_confound[f] = {"r_ops": r_ops, "r_depth": r_depth, "n": int(m.sum())}
    print(f"  {f:8s} r(ops) = {r_ops:+.3f}   r(depth) = {r_depth:+.3f}   (n = {m.sum()})")

story_mask = forms == "story"
fig, axes = plt.subplots(2, len(PLOT_REPS), figsize=(4.2 * len(PLOT_REPS), 8), squeeze=False)
fig.suptitle("Data-space PCA by total operation count (stories only)")
for col, name in enumerate(PLOT_REPS):
    Xs = REPS[name][story_mask]
    for row, (X, label) in enumerate(((Xs, "raw"),
                                      (residualize(Xs, lengths[story_mask]), "length residualized"))):
        X2, evr = pca2(X)
        sc = axes[row][col].scatter(X2[:, 0], X2[:, 1], c=ops[story_mask], s=14,
                                   cmap="viridis", alpha=0.85)
        rho = spearmanr(X2[:, 0], ops[story_mask]).correlation
        axes[row][col].set_title(f"{name}, {label}\n{evr:.0%} var, PC1 vs ops rho = {rho:+.2f}",
                                 fontsize=9)
        axes[row][col].set_xticks([])
        axes[row][col].set_yticks([])
fig.colorbar(sc, ax=axes.ravel().tolist(), shrink=0.6, label="total ops")
plt.show()

## C — Law identity: the floor the semantic-core test has to clear

Experiment 3's headline claim is that renderings of the *same law* co-locate across surface
forms. Here is how much of that is already available without a model, measured with experiment
3's own metric so the numbers are directly comparable:

- **all surfaces** — nearest neighbour must come from a different surface group;
- **story × theme** — story queries, story candidates from other themes only, i.e. the
  disjoint-vocabulary version.

Two rows deserve attention before the interesting ones. The `symbolic` row is not a floor but the
ceiling — it retrieves perfectly by construction, and is here only to confirm the metric measures
what we think. The `length only` row is the row that matters: because the renderers are templates,
a text's word count is close to a fingerprint of its law, so **length alone is a serious retriever**
and the headline floor is whichever surface representation scores highest, not the TF-IDF one by
default.

The second table reports the same metrics after `residualize(X, lengths)`, which is the version to
compare a length-residualized activation result against. The third shows *where* surface leakage
lives — same-law minus different-law mean similarity, form pair by form pair.

In [ ]:
# ---------------------- C: retrieval floors and where leakage lives ----------------------

def retrieval_row(X):
    return {"all": law_retrieval_all(X), "story_x_theme": law_retrieval_story(X)}


floors = {name: retrieval_row(X) for name, X in REPS.items()}
floors_residualized = {name: retrieval_row(residualize(REPS[name], lengths))
                       for name in TEXT_REPS + ["symbolic"]}

print(f"1-NN law retrieval  (chance = 1/{n_pairs} = {CHANCE:.1%})")
print(f"  {'representation':18s} {'all surfaces':>13s} {'story x-theme':>14s}")
for name, f in floors.items():
    tag = {"symbolic": "  <- ceiling, by construction",
           "length only": "  <- template artifact, not semantics"}.get(name, "")
    print(f"  {name:18s} {f['all']:>12.1%} {f['story_x_theme']:>14.1%}{tag}")

surface_floor = {metric: max(f[metric] for name, f in floors.items() if name != "symbolic")
                 for metric in ("all", "story_x_theme")}
driver = {metric: max((n for n in floors if n != "symbolic"), key=lambda n: floors[n][metric])
          for metric in surface_floor}
print(f"\n=> floor experiment 03 must beat: all surfaces {surface_floor['all']:.1%} "
      f"(set by '{driver['all']}'), story x-theme {surface_floor['story_x_theme']:.1%} "
      f"(set by '{driver['story_x_theme']}')")

print("\nsame metrics after residualizing word count out of the representation:")
print(f"  {'representation':18s} {'all surfaces':>13s} {'story x-theme':>14s}")
for name, f in floors_residualized.items():
    print(f"  {name:18s} {f['all']:>12.1%} {f['story_x_theme']:>14.1%}")

FORM_PAIRS = [("story", "story"), ("story", "literal"), ("story", "rg"),
              ("literal", "rg")]
print("\nsame-law minus different-law mean similarity, by form pair (word TF-IDF / char TF-IDF):")
leakage = {}
for name in ("word TF-IDF", "char TF-IDF"):
    S = similarity(REPS[name])
    for fa, fb in FORM_PAIRS:
        block = (forms[:, None] == fa) & (forms[None, :] == fb)
        block &= surfaces[:, None] != surfaces[None, :]      # cross-surface only
        same = block & (pair_ids[:, None] == pair_ids[None, :])
        diff = block & (pair_ids[:, None] != pair_ids[None, :])
        if not same.any():
            continue
        gap = float(S[same].mean() - S[diff].mean())
        leakage[f"{name}|{fa}-{fb}"] = gap
        print(f"  {name:12s} {fa:8s} vs {fb:8s}  same {S[same].mean():.3f}  "
              f"diff {S[diff].mean():.3f}  gap {gap:+.3f}")

fig, ax = plt.subplots(figsize=(7.5, 4))
names = [n for n in floors if n != "symbolic"]
x = np.arange(len(names))
ax.bar(x - 0.2, [floors[n]["all"] for n in names], 0.4, label="all surfaces")
ax.bar(x + 0.2, [floors[n]["story_x_theme"] for n in names], 0.4, label="story x-theme")
ax.axhline(CHANCE, color="gray", ls=":", lw=1, label=f"chance {CHANCE:.1%}")
ax.axhline(surface_floor["story_x_theme"], color="red", ls="--", lw=1,
           label=f"floor to beat {surface_floor['story_x_theme']:.1%}")
ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=9)
ax.set_ylabel("1-NN law retrieval")
ax.set_title("Surface floors experiment 03 must beat")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## D — The target geometry: what structure do the labels actually have?

The precondition for the whole PCA programme. Days of the week lie on a circle because the labels
are ordered; if our 52 laws are mutually unrelated points, then "activations cluster by law" is
the only law-related question the dataset can answer, and anything about the geometry *among*
laws (a manifold of semantic content, nearby laws nearby in activation space) is untestable here
no matter what the model does.

Two independent notions of law similarity, so the answer does not depend on one arbitrary metric:
the subtree-bag cosine, and normalized edit distance between the alpha-renamed canonical term
strings. If they disagree, there is no metric-independent notion of "similar law" in this sample.

In [ ]:
# ---------------------- D: structure among the laws themselves ----------------------

law_ops = np.array([s["ops_total"] for s in kept_pairs], dtype=float)
law_depth = np.array([s["depth"] for s in kept_pairs], dtype=float)

G = LAW_SYMBOLIC @ LAW_SYMBOLIC.T
off = ~np.eye(n_pairs, dtype=bool)
distinct = len({tuple(np.round(r, 6)) for r in LAW_SYMBOLIC})
print(f"subtree cosine between distinct laws: mean {G[off].mean():.3f}  "
      f"min {G[off].min():.3f}  max {G[off].max():.3f}")
print(f"{distinct}/{n_pairs} laws have a distinct symbolic vector")


def canonical_term_string(meta):
    return "|".join(alpha_shape(t) for key in ("equation_e", "equation_f")
                    for t in parse_equation(meta[key]))


def levenshtein(a, b):
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i] + [0] * len(b)
        for j, cb in enumerate(b, 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb))
        prev = cur
    return prev[-1]


term_strings = [canonical_term_string(s["metadata"]) for s in kept_pairs]
D = np.zeros((n_pairs, n_pairs))
for i in range(n_pairs):
    for j in range(i + 1, n_pairs):
        D[i, j] = D[j, i] = levenshtein(term_strings[i], term_strings[j]) / max(
            len(term_strings[i]), len(term_strings[j]))
print(f"normalized term edit distance: mean {D[off].mean():.3f}  "
      f"min {D[off].min():.3f}  max {D[off].max():.3f}")

tri = np.triu(np.ones((n_pairs, n_pairs), dtype=bool), 1)
agreement = spearmanr(G[tri], -D[tri]).correlation
print(f"\nagreement between the two law-similarity notions: rho = {agreement:+.3f}")
print(f"edit distance vs |ops difference|:                 rho = "
      f"{spearmanr(D[tri], np.abs(law_ops[:, None] - law_ops[None, :])[tri]).correlation:+.3f}")

law_pca = PCA(n_components=3).fit(LAW_SYMBOLIC)
P = law_pca.transform(LAW_SYMBOLIC)
print(f"\nsymbolic PCA over laws: top-2 explains {law_pca.explained_variance_ratio_[:2].sum():.1%}, "
      f"top-3 {law_pca.explained_variance_ratio_[:3].sum():.1%}")
for j in range(3):
    print(f"  PC{j + 1}: r(ops) = {np.corrcoef(P[:, j], law_ops)[0, 1]:+.3f}   "
          f"r(depth) = {np.corrcoef(P[:, j], law_depth)[0, 1]:+.3f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.6))
sc = ax1.scatter(P[:, 0], P[:, 1], c=law_ops, s=45, cmap="viridis")
ax1.set_title("laws in symbolic space, colored by ops")
fig.colorbar(sc, ax=ax1, shrink=0.8, label="total ops")
ax2.hist(G[tri], bins=30, alpha=0.75, label="subtree cosine")
ax2.hist(1 - D[tri], bins=30, alpha=0.75, label="1 - edit distance")
ax2.set_title("pairwise law similarity: is anything close to anything?")
ax2.legend(fontsize=8)
for ax in (ax1,):
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

## E — Exports for experiment 3

Two artifacts.

**`baselines.json`** — the floors, the ladder statistic, and the length confound, so experiment 3
can draw them as reference lines instead of hard-coding numbers.

**`similarity-matrices.npz`** — the text-level similarity matrices, which turn experiment 3's
eyeball plots into one number via representational similarity analysis. With an activation
similarity matrix `S_act`, the quantity of interest is the partial Spearman correlation between
`S_act` and `symbolic`, holding the surface floor fixed:

```python
r_ab_c = (r_ab - r_ac * r_bc) / sqrt((1 - r_ac ** 2) * (1 - r_bc ** 2))
```

with a = activations, b = symbolic, c = word TF-IDF, all correlations taken over cross-surface
text pairs (the `cross_surface_mask` in the archive). That is "how much does this layer's
geometry track the law structure, beyond what the wording already gives away" — one number per
layer, which is the claim experiment 3 wants to make.

In [ ]:
# ---------------------- E: save artifacts ----------------------

cross_surface = np.triu(np.ones((len(records), len(records)), dtype=bool), 1) & (
    surfaces[:, None] != surfaces[None, :])

baselines = {
    "texts_sha": hashlib.sha256(json.dumps(texts).encode()).hexdigest()[:12],
    "config": {"seed": SEED, "per_bin": PER_BIN, "story_themes": STORY_THEMES,
               "repo_commit": REPO_COMMIT, "char_ngram_range": list(CHAR_NGRAM_RANGE)},
    "n_pairs": n_pairs, "n_texts": len(records), "chance": CHANCE,
    "law_retrieval_floors": floors,
    "law_retrieval_floors_length_residualized": floors_residualized,
    "surface_floor": surface_floor,
    "surface_floor_driver": driver,
    "ladder_statistic": ladder,
    "length_confound": length_confound,
    "form_pair_leakage": leakage,
    "law_similarity": {
        "subtree_cosine_mean": float(G[off].mean()),
        "subtree_cosine_max": float(G[off].max()),
        "edit_distance_mean": float(D[off].mean()),
        "edit_distance_min": float(D[off].min()),
        "metric_agreement_rho": float(agreement),
        "distinct_symbolic_vectors": int(distinct),
    },
}
(OUT_DIR / "baselines.json").write_text(json.dumps(baselines, indent=2))

np.savez_compressed(
    OUT_DIR / "similarity-matrices.npz",
    cross_surface_mask=cross_surface,
    symbolic=similarity(REPS["symbolic"]),
    word_tfidf=similarity(REPS["word TF-IDF"]),
    char_tfidf=similarity(REPS["char TF-IDF"]),
    length=similarity(REPS["length only"]),
    pair_ids=pair_ids, forms=forms, surfaces=surfaces, ops=ops, depth=depth, lengths=lengths,
)
print(f"wrote {OUT_DIR / 'baselines.json'}")
print(f"wrote {OUT_DIR / 'similarity-matrices.npz'}")
print(json.dumps(baselines["law_retrieval_floors"], indent=2))

## Interpretation notes

Fill in after the run; the run on the default config (`SEED = 0`, `PER_BIN = 10`, 52 pairs)
gave the following, which is what the decision mapping below assumes.

- **A (form).** Word TF-IDF already separates the three forms (~38% of variance in two PCs), so
  form separation in activations is uninformative on its own. The ladder statistic, though, says
  the forms are *not* a path in text space: the literal centroid projects at t ≈ 0.17 on the
  story→RG axis with an off-axis residual ≈ 0.9 × the axis length, i.e. a triangle (char n-grams
  agree: t ≈ 0.22, residual ≈ 0.95). Experiment 7's H1 has a clean null to beat, and betweenness
  in activation space would be a real finding.
- **B (complexity).** Fatal as currently written. Within literal and RG, `ops_total` is a
  *deterministic* function of word count (r = 1.000); within stories r = 0.928, and `depth` is
  confounded too (r = 0.57–0.70). Any complexity gradient found in activations is a length
  gradient until shown otherwise. Fix upstream by padding low-op instances with distractor detail
  to match lengths, or at minimum apply `residualize(X, lengths)` before the PCA and report both
  panels.
- **C (law identity).** Well-posed, but the floor is much higher than the lexical baselines
  suggest, and this is the main thing this notebook changes. Against chance of 1.9%, word TF-IDF
  retrieves at 3.5% / 4.8% (all-surface / story-cross-theme) and char n-grams at 6.7% / 9.1% —
  but **word count alone retrieves at 15.4% / 24.5%**. Because the renderers are templates, a
  text's length is nearly a fingerprint of its law, so a mean-pooled activation vector gets a
  strong law signal for free from sequence length. Experiment 3's semantic-core claim has to clear
  ~25% on the story-cross-theme metric, not ~7% — or be reported on length-residualized
  activations, in which case the floor is the residualized 12.5%. Note also that essentially all
  *lexical*
  leakage is story↔story (gap +0.015); story↔RG and literal↔RG share no vocabulary at all
  (gap 0.000), so cross-form retrieval between the abstract forms needs no lexical control.
- **D (target geometry).** The 52 laws are close to mutually orthogonal in symbolic space
  (IDF-weighted subtree cosine: mean 0.06) and the two similarity notions barely agree
  (rho ≈ +0.08). There is no graded "law similarity" structure in this sample, and the symbolic
  PCs do not track ops or depth (|r| ≤ 0.36).

**Decision mapping for experiments 3 and 7:**

- *Add the length control to experiment 3 before trusting any of its results.* Concretely: report
  1-NN law retrieval on `residualize(X, lengths)` alongside the raw number, and draw the 24.5%
  length floor on the layer-sweep figure. Without it, a "semantic core" curve that peaks around
  20% is indistinguishable from the model encoding how long the input was.
- *Law **identity** is testable; law **geometry** is not.* Asking whether the six renderings of one
  law co-locate is fine. Asking whether similar laws land near each other — the "manifold of
  semantic content" framing — needs a different sample: laws drawn in families (one law plus
  systematic perturbations) rather than stratified by operation count. Worth deciding before
  spending GPU time on a larger run.
- *The formality axis is the only ordered axis this dataset has*, which is consistent with the
  proposal's own claim that the formality ladder is the right object to fit a path to. Keep
  experiment 7 pointed there, and use the section-A ladder statistic as its null.
- *If a later change to the renderers or sampler moves the `texts SHA`*, rerun this notebook — the
  floors are dataset-specific and cost a minute.